# 05 · Exportación para Power BI (OE4)

Fase 6 de CRISP-DM. Genera el archivo `powerbi/tablero_desercion.xlsx` con todo lo que necesita el tablero: los estudiantes con etiquetas en español, el puntaje de riesgo fuera de muestra en los tres momentos con sus tres factores principales, la calibración del puntaje, las métricas del modelo, la importancia SHAP y la asociación de la Fase 2.

**Tiempo estimado en Colab: entre 3 y 8 minutos.**

In [ ]:
# Preparar el entorno (igual que en el cuaderno 00)
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"
if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import os, warnings
os.environ["PYTHONWARNINGS"] = "ignore:Unknown solver options"
warnings.filterwarnings("ignore", message="Unknown solver options")

import pandas as pd
from src import config, prep, export
from src.data import load_raw, binarize_target
from src.utils import set_seed

set_seed()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
df = prep.add_no_enrollment_flags(binarize_target(load_raw()))
idx_train, idx_test = prep.load_split()

## 1. Generar el archivo para Power BI

Para cada momento, el modelo final puntúa a cada estudiante sin haberlo visto: los del entrenamiento con validación cruzada de 5 pliegues y los de la prueba con el modelo entrenado en todo el entrenamiento. Con el mismo modelo se calculan los valores SHAP y se guardan las tres variables que más suben el riesgo de cada estudiante.

In [ ]:
ruta, hojas = export.export_workbook(df, idx_train, idx_test)
print("Archivo:", ruta)
for nombre, t in hojas.items():
    print(f"  {nombre:16s} {t.shape[0]:6d} filas × {t.shape[1]:2d} columnas")

## 2. Verificación

Las métricas recalculadas a partir de la hoja `Predicciones` (solo prueba) deben coincidir con las de la Fase 5.

In [ ]:
pred = hojas["Predicciones"]
prueba = pred[pred["Origen"] == "Prueba"]
for m, g in prueba.groupby("Momento", sort=False):
    alerta = g["Alerta"] == "Sí"
    vp = int((alerta & (g["Desertó"] == 1)).sum())
    print(f"{m:22s} sensibilidad = {vp / g['Desertó'].sum():.3f} | precisión = {vp / alerta.sum():.3f}")
print()
print(hojas["Metricas"][["Momento", "Sensibilidad", "Precisión"]].round(3).to_string(index=False))

## 3. Resumen para el documento

Ejecutar esta celda y enviar su salida completa.

In [ ]:
print("CONTROL (valores que el tablero debe mostrar):")
print(hojas["Control"].to_string(index=False))
print("\nCALIBRACIÓN DEL PUNTAJE EN PRUEBA (cierre 2.º semestre):")
c = hojas["Calibracion"]
print(c[c["Orden momento"] == 3].sort_values("Orden rango")[["Rango de puntaje", "Estudiantes", "Desertores", "Deserción observada (%)"]].to_string(index=False))
print("\nEJEMPLO DE PREDICCIONES CON ALERTA (cierre 2.º semestre, prueba):")
e = prueba[(prueba["Orden momento"] == 3) & (prueba["Alerta"] == "Sí")].sort_values("Puntaje de riesgo", ascending=False).head(5)
print(e[["ID", "Puntaje de riesgo", "Desertó", "Factor 1", "Factor 2", "Factor 3"]].to_string(index=False))

## 4. Guardar en GitHub y descargar para Power BI

La siguiente celda empaqueta las carpetas `powerbi/` y `reports/` en `fase6.zip`. Descárgalo y descomprímelo: el archivo `powerbi/tablero_desercion.xlsx` es el que vas a abrir en Power BI. Sube la carpeta `powerbi` completa al repositorio con el mensaje `Fase 6: datos para Power BI`, y guarda este cuaderno con Archivo > Guardar una copia en GitHub.

In [ ]:
!cd /content/tesis-desercion && zip -qr /content/fase6.zip powerbi reports && echo "Listo: /content/fase6.zip"